# Q10.
```{admonition}
:class: note
We showed how to fit a linear AR model to the `NYSE` data using the `LinearRegression()` function. However, we also mentioned that we can "flatten" the short sequences produced for the RNN model in order to fit a linear AR model. Use this latter approach to fit a linear AR model to the `NYSE` data. Compare the test $R^{2}$ of this linear AR model to that of the linear AR model that we fit in the lab. What are the advantages/disadvantages of each approach?

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [12]:
import torch
from torchinfo import summary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
nyse = pd.read_csv('../../../ALL CSV FILES - 2nd Edition/NYSE.csv', parse_dates=['date'], index_col=['date']).sort_index().drop(columns=['day_of_week','train'])

In [5]:
nyse_train, nyse_test = train_test_split(nyse, shuffle=False, train_size=4276)

In [6]:
scaler = StandardScaler()
nyse_train[nyse.columns] = scaler.fit_transform(nyse_train)
nyse_test[nyse.columns] = scaler.transform(nyse_test)
nyse_scaled = pd.concat([nyse_train,nyse_test])
nyse_scaled.head(3)

,DJ_return,log_volume,log_volatility
date,,,
1962-12-03,-0.559615,0.193763,-3.982432
1962-12-04,0.959704,1.552668,-2.240505
1962-12-05,0.468531,2.328698,-2.134713


In [7]:
nyse_lag = pd.concat([nyse_scaled]+[nyse_scaled.shift(lag).add_suffix(f'_lag{lag}') for lag in range (1,6)], axis=1).dropna()
nyse_lag.head(3)

,DJ_return,log_volume,log_volatility,DJ_return_lag1,log_volume_lag1,log_volatility_lag1,DJ_return_lag2,log_volume_lag2,log_volatility_lag2,DJ_return_lag3,log_volume_lag3,log_volatility_lag3,DJ_return_lag4,log_volume_lag4,log_volatility_lag4,DJ_return_lag5,log_volume_lag5,log_volatility_lag5
date,,,,,,,,,,,,,,,,,,
1962-12-10,-1.347250,0.629963,-1.132250,0.062892,0.244084,-2.213740,-0.435955,0.963315,-2.085623,0.468531,2.328698,-2.134713,0.959704,1.552668,-2.240505,-0.559615,0.193763,-3.982432
1962-12-11,0.007932,0.002680,-1.265313,-1.347250,0.629963,-1.132250,0.062892,0.244084,-2.213740,-0.435955,0.963315,-2.085623,0.468531,2.328698,-2.134713,0.959704,1.552668,-2.240505
1962-12-12,0.408248,0.059592,-1.309001,0.007932,0.002680,-1.265313,-1.347250,0.629963,-1.132250,0.062892,0.244084,-2.213740,-0.435955,0.963315,-2.085623,0.468531,2.328698,-2.134713


In [8]:
Y = nyse_lag['log_volume'].to_numpy().astype(np.float32)
X = nyse_lag.drop(columns=nyse_scaled.columns).to_numpy().astype(np.float32)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, shuffle=False, train_size=4276)

In [10]:
class NYSEModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.ar = nn.Linear(15,1)

    def forward(self, x):
        return self.ar(x)

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_t = torch.tensor(X_train,dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32)
nyse_train = TensorDataset(X_train_t,y_train_t)

train_loader = DataLoader(
    nyse_train,
    batch_size=64,
    shuffle=False
)

X_test_t = torch.tensor(X_test,dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test,dtype=torch.float32).to(device)

In [14]:
nyse_model = NYSEModel().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(nyse_model.parameters(),lr=0.01)
epochs = 50

for epoch in range(epochs):
    nyse_model.train()

    for X_batch, y_batch in train_loader:
        mses = nyse_model(X_batch.to(device)).squeeze()
        loss = criterion(mses,y_batch.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    nyse_model.eval()
    with torch.no_grad():
        preds = nyse_model(X_test_t)
        mse = mean_squared_error(y_test,preds.cpu().numpy())
        r2 = r2_score(y_test,preds.cpu().numpy())
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: mse={mse:.4f}, r2={r2:.4f}')

Epoch 10: mse=0.6417, r2=0.4058
Epoch 20: mse=0.6379, r2=0.4094
Epoch 30: mse=0.6367, r2=0.4105
Epoch 40: mse=0.6363, r2=0.4109
Epoch 50: mse=0.6361, r2=0.4110


In [15]:
nyse_model.eval()

with torch.no_grad():
    pred = nyse_model(X_test_t)
    mse = mean_squared_error(y_test,preds.cpu().numpy())
    r2 = r2_score(y_test,preds.cpu().numpy())
print(f'Test MSE={mse:.4f}, Test R2={r2:.4f}')

Test MSE=0.6361, Test R2=0.4110
